Here is the code where you put in H, M, G, model and measurement

In [1]:
import pickle
import pandas as pd 
from anthropmass.data_module import *
from anthropmass.anthro_module import *
from anthropmass.bambi_module import *

WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install m2w64-toolchain`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


Normalize is in data module but i dont know how to import

This function is normalizing the persons weight and height

In [2]:
def normalize_person(weight, height, gender):
    person=pd.DataFrame({'weightkg': [weight], 'stature': [height], 'Gender': [gender]})
    normalize(person, 'weightkg')
    normalize(person, 'stature')
    return person

This functions gets the pickled model

In [3]:
def get_pickled_model(kindofmodel:str, measurement:str):
    filepath = f'../output/anthro_models/{kindofmodel}/pickle_{measurement}_{kindofmodel}'
    with open(filepath,'rb') as file:
        model=pickle.load(file)
    return model

Predict_from_model uses the pickled model and the normalized person to predict the measurement

In [4]:
def predict_from_model(kindofmodel:str, measurement:str, w, h, g):
    pickledmodel = get_pickled_model(kindofmodel, measurement)
    person = normalize_person(w, h, g)
    if kindofmodel=='xgboost':
        return pickledmodel.predict(person)
    elif kindofmodel=='bambi':
        model = model_bmb(measurement)
        return predict_mean_bmb(pickledmodel, model, person, measurement)
    elif kindofmodel=='bambi_ml_gc':
        train=pd.read_csv('../data/processed/ANSURIInormalizedtrain.csv')
        #setpriors_com(measurement,train)
        model = component_model(measurement,train)
        return predict_mean_bmb(pickledmodel, model, person, measurement)
    else:
        return 'wrong model name'

In [5]:
def add_to_df(df, measurement, pred):
    df[measurement]=pred
    return df

I dont know if we need to make it to csv?

In [6]:
def make_csv(data, name):
    data.to_csv(f'{name}.csv')

Here are all predictions made, the measurements are in a list

In [7]:
def make_predictions(kindofmodel:str, measurements:list, w, h, g):
    output=pd.DataFrame()
    for m in measurements:
        pred = predict_from_model(kindofmodel, m, w, h, g)
        add_to_df(output, m, pred)
    return output

In [8]:
make_predictions('bambi_ml_gc',['abdominalextensiondepthsitting','acromialheight'], 80, 1800, 1)

KeyError: 'Component'

In [9]:
make_predictions('xgboost',['abdominalextensiondepthsitting','acromialheight'], 80, 1800, 1)

,abdominalextensiondepthsitting,acromialheight
0,232.044022,1474.543213
